# Learning 02: PII Detection and Anonymization

**PII** (Personally Identifiable Information) is any data that can identify an individual.
Regulations like GDPR, HIPAA, and PCI-DSS require that PII is handled carefully — often
redacted from logs, anonymized in datasets, or masked before sharing.

**`gliner-pii-edge-v1.0`** is a GLiNER model fine-tuned specifically for 60+ PII categories:
- Personal: name, DOB, gender, age
- Contact: email, phone, address, IP
- Financial: credit card, SSN, bank account, routing number
- Healthcare: condition, drug, medical record number
- Identity docs: passport, driver license, username, password

Because it's a GLiNER model, you also get zero-shot flexibility — specify exactly the
PII types relevant to your domain.

## Setup

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "pii_documents.json")) as f:
    documents = json.load(f)

print(f"✓ Loaded {len(documents)} documents")
print(f"  Sample: {documents[0]['text'][:90]}...")

✓ Loaded 8 documents
  Sample: Customer John Smith called our support line from 415-555-0182 on March 10, 2024. His email...


## 1. Load the PII Model

In [2]:
pii_model = GLiNER.from_pretrained("knowledgator/gliner-pii-edge-v1.0")
print("✓ PII model loaded")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✓ PII model loaded


## 2. Basic PII Detection

The API is identical to standard GLiNER — just use PII-specific label names.

In [3]:
text = documents[0]["text"]  # Support ticket
labels = ["name", "phone number", "email address", "account number"]

entities = pii_model.predict_entities(text, labels, threshold=0.3)

print(f"Text: {text}\n")
print(f"Found {len(entities)} PII entities:")
for e in entities:
    print(f"  [{e['label']:20}] {e['text']!r:30} score={e['score']:.2f}")

Text: Customer John Smith called our support line from 415-555-0182 on March 10, 2024. His email is john.smith@example.com and he reported an issue with his account number 789-654-3210.

Found 4 PII entities:
  [name                ] 'John Smith'                   score=0.89
  [phone number        ] '415-555-0182'                 score=0.69
  [email address       ] 'john.smith@example.com'       score=0.30
  [account number      ] '789-654-3210'                 score=0.68


## 3. Full PII Label Set

Using all 60+ categories gives comprehensive coverage.

In [4]:
ALL_PII_LABELS = [
    # Personal
    "name", "first name", "last name", "dob", "age", "gender",
    # Contact
    "email address", "phone number", "ip address", "url",
    "location address", "location city", "location state", "location country", "location zip",
    # Financial
    "account number", "bank account", "routing number",
    "credit card", "credit card expiration", "cvv", "ssn", "money",
    # Healthcare
    "condition", "medical process", "drug", "dose",
    "organization medical facility", "healthcare number", "medical code",
    "name medical professional",
    # Identity
    "passport number", "driver license", "username", "password", "vehicle id",
]

# HR record — should find many PII types
hr_text = documents[1]["text"]
entities = pii_model.predict_entities(hr_text, ALL_PII_LABELS, threshold=0.3)

print(f"HR record: {hr_text}\n")
print(f"Found {len(entities)} entities:")
for e in entities:
    print(f"  [{e['label']:20}] {e['text']!r}")

HR record: Employee record: Sarah Johnson, DOB 04/15/1985, SSN 123-45-6789. Home address: 742 Evergreen Terrace, Springfield, IL 62701. Emergency contact: Michael Johnson at 312-555-9988.

Found 8 entities:
  [first name          ] 'Sarah'
  [last name           ] 'Johnson'
  [dob                 ] '04/15/1985'
  [ssn                 ] '123-45-6789'
  [location address    ] '742 Evergreen Terrace, Springfield, IL 62701'
  [first name          ] 'Michael'
  [last name           ] 'Johnson'
  [phone number        ] '312-555-9988'


## 4. Anonymization: Replace PII with Placeholders

The core anonymization pattern:
1. Detect PII entities (get start/end offsets)
2. Sort from **end to start** (to preserve earlier offsets while replacing later ones)
3. Replace each span with `<LABEL_TYPE>` placeholder

In [5]:
def anonymize_text(model, text, labels, threshold=0.3):
    """Replace detected PII spans with <LABEL> placeholders."""
    entities = model.predict_entities(text, labels, threshold=threshold)

    # Critical: sort from end to start so early offsets stay valid
    entities.sort(key=lambda e: e["start"], reverse=True)

    anonymized = text
    for e in entities:
        placeholder = "<" + e["label"].upper().replace(" ", "_") + ">"
        anonymized = anonymized[:e["start"]] + placeholder + anonymized[e["end"]:]

    return anonymized


pii_labels = ["name", "phone number", "email address", "account number"]
original = documents[0]["text"]
anonymized = anonymize_text(pii_model, original, pii_labels)

print("Original: ", original)
print()
print("Anonymized:", anonymized)

Original:  Customer John Smith called our support line from 415-555-0182 on March 10, 2024. His email is john.smith@example.com and he reported an issue with his account number 789-654-3210.

Anonymized: Customer <NAME> called our support line from <PHONE_NUMBER> on March 10, 2024. His email is <EMAIL_ADDRESS> and he reported an issue with his account number <ACCOUNT_NUMBER>.


## 5. Why Sort End-to-Start?

If we replace spans left-to-right, each replacement changes the length of the string,
making all subsequent offsets wrong. Replacing right-to-left avoids this.

In [6]:
# Demonstrate the offset problem
text = "John Smith, john@example.com"
#       0         1         2
#       0123456789012345678901234567

# Suppose entities: John Smith [0:10], john@example.com [12:28]
# Replace John Smith first → '<NAME>, john@example.com'
# Now john@example.com is at offset 7, not 12!

# Right-to-left: replace john@example.com first → 'John Smith, <EMAIL_ADDRESS>'
# Then replace John Smith → '<NAME>, <EMAIL_ADDRESS>'  ✓

print("Always replace from end to start to preserve offset validity")
print(f"String: {text!r}")
print(f"After right-to-left replacement: '<NAME>, <EMAIL_ADDRESS>'")

Always replace from end to start to preserve offset validity
String: 'John Smith, john@example.com'
After right-to-left replacement: '<NAME>, <EMAIL_ADDRESS>'


## 6. Batch Anonymization

In [7]:
def anonymize_batch(model, texts, labels, threshold=0.3, batch_size=8):
    """Anonymize multiple documents efficiently."""
    # Get all predictions in one batch call
    all_entities = model.inference(texts, labels, threshold=threshold, batch_size=batch_size)

    results = []
    for text, entities in zip(texts, all_entities):
        entities_sorted = sorted(entities, key=lambda e: e["start"], reverse=True)
        anonymized = text
        for e in entities_sorted:
            placeholder = "<" + e["label"].upper().replace(" ", "_") + ">"
            anonymized = anonymized[:e["start"]] + placeholder + anonymized[e["end"]:]
        results.append(anonymized)

    return results


pii_labels = [
    "name", "phone number", "email address", "ssn",
    "credit card", "location address", "dob", "account number",
    "passport number", "driver license", "password", "username",
]

texts = [doc["text"] for doc in documents]
anonymized_docs = anonymize_batch(pii_model, texts, pii_labels)

for i, (orig, anon) in enumerate(zip(texts, anonymized_docs)):
    print(f"Doc {i+1} [{documents[i]['domain']:12}]:")
    print(f"  Before: {orig[:80]}...")
    print(f"  After:  {anon[:80]}...")
    print()

Doc 1 [support     ]:
  Before: Customer John Smith called our support line from 415-555-0182 on March 10, 2024....
  After:  Customer <NAME> called our support line from <SSN> on March 10, 2024. His email ...

Doc 2 [hr          ]:
  Before: Employee record: Sarah Johnson, DOB 04/15/1985, SSN 123-45-6789. Home address: 7...
  After:  Employee record: <NAME>, DOB <DOB>, SSN <SSN>. Home address: <LOCATION_ADDRESS>....

Doc 3 [healthcare  ]:
  Before: Patient: Robert Davis, Date of Birth: 02/28/1972. Admitted to St. Mary's Medical...
  After:  Patient: <NAME>, Date of Birth: <DOB>. Admitted to St. Mary's Medical Center on ...

Doc 4 [finance     ]:
  Before: Transaction alert: A charge of $2,499.00 was made on credit card ending 4532 (ex...
  After:  Transaction alert: A charge of $2,499.00 was made on credit card ending <CREDIT_...

Doc 5 [insurance   ]:
  Before: Insurance claim submitted by Jennifer White, policy number POL-2024-88732. Claim...
  After:  Insurance claim submitted by <

## 7. Threshold Strategy for PII

For compliance use cases, **recall matters more than precision** — a missed SSN is worse
than an over-redacted word. Use lower thresholds (0.2–0.3) and review edge cases.

In [8]:
medical_text = documents[2]["text"]
healthcare_labels = [
    "name", "dob", "organization medical facility",
    "condition", "name medical professional",
]

print(f"Text: {medical_text}\n")
for threshold in [0.2, 0.3, 0.5]:
    entities = pii_model.predict_entities(medical_text, healthcare_labels, threshold=threshold)
    print(f"  threshold={threshold}: {len(entities)} entities")
    for e in entities:
        print(f"    [{e['label']:30}] {e['text']!r:25} {e['score']:.2f}")
    print()

Text: Patient: Robert Davis, Date of Birth: 02/28/1972. Admitted to St. Mary's Medical Center on January 5, 2024. Diagnosis: Type 2 diabetes. Insurance policy: BlueCross-1234567. Physician: Dr. Emily Carter.



  threshold=0.2: 5 entities
    [name                          ] 'Robert Davis'            0.79
    [dob                           ] '02/28/1972'              0.83
    [organization medical facility ] "St. Mary's Medical Center" 0.72
    [condition                     ] 'Type 2 diabetes'         0.74
    [name                          ] 'Dr. Emily Carter'        0.71



  threshold=0.3: 5 entities
    [name                          ] 'Robert Davis'            0.79
    [dob                           ] '02/28/1972'              0.83
    [organization medical facility ] "St. Mary's Medical Center" 0.72
    [condition                     ] 'Type 2 diabetes'         0.74
    [name                          ] 'Dr. Emily Carter'        0.71

  threshold=0.5: 5 entities
    [name                          ] 'Robert Davis'            0.79
    [dob                           ] '02/28/1972'              0.83
    [organization medical facility ] "St. Mary's Medical Center" 0.72
    [condition                     ] 'Type 2 diabetes'         0.74
    [name                          ] 'Dr. Emily Carter'        0.71



## Summary

- `gliner-pii-edge-v1.0` detects 60+ PII categories using zero-shot GLiNER
- Same API: `predict_entities(text, labels, threshold)` → `[{text, label, start, end, score}]`
- Anonymization: detect → sort end-to-start → replace spans with `<LABEL>` placeholders
- For compliance (GDPR/HIPAA): prefer lower thresholds (0.2–0.3) to maximize recall
- `model.inference(texts, labels, batch_size)` for efficient batch anonymization